# The Fermi-Pasta-Ulam-Tsingou problem

Jacopo Tissino  
June 9, 2026

In [1]:
import matplotlib.pyplot as plt
from project_scripts.simulations import non_linear_system
from pathlib import Path
import numpy as np

fname = Path('alpha_5.mp4')

if not fname.exists():

    non_linear_system(
            np.sin(np.pi*np.linspace(0, 1, 21)),
            periods=10,
            alpha=1,
            video_path=fname,
            label=', $\\alpha=5$',
        )
    plt.close()

## Motivation and history

To write. Mention “the footnote” about Mary Tsingou.

## Linear problem description

Consider a horizontal chain of masses connected by springs. Each of them can oscillate in the vertical direction. We will use “horizontal” and “vertical” for clarity, but we are neglecting gravity here.

Hooke’s law tells us that force is related to displacement through $\vec{F} = -k \vec{x}$; for the $i$-th mass, the horizontal component of this force will always be balanced (since the horizontal distance to its neighboring masses is fixed).

We can parameterize these masses through their vertical coordinates; the $i$-th mass will be subject to a force

$$ F_i = -k (x_{i} - x_{i-i}) - k (x_{i} - x_{i+1})
= k ( x_{i-1} + x_{i+1} - 2 x_i )$$

meaning that its acceleration, through $F = m a$, will be

$$ \ddot{x}_i = \frac{k}{m} ( x_{i-1} + x_{i+1} - 2 x_i )\,.$$

We consider a system of $N+1$ masses, indexed from $0$ to $N$, such that masses $0$ and $N$ are kept fixed.

We expect wave-like solutions by the look of the equation. Since we need to fix the boundaries to 0, a sensible family to consider is

$$ x_i^{(n)}(t) = Q_n(t) \sin\left( \frac{\pi i n}{N}\right)\,.$$

Are these indeed solutions? Plugging them into our ODE we get

$$ \frac{\text{d}^2}{\text{d}t^2} x_i^{(n)}(t) = \ddot{Q}_n(t) \sin\left( \frac{\pi i n}{N}\right) = (*)\,.$$

On the other side, we have

$$ (*) = Q_n(t) \left[ \sin\left( \frac{\pi (i+1) n}{N}\right) + \sin\left( \frac{\pi (i-1) n}{N}\right) - 2 \sin\left( \frac{\pi i n}{N}\right) \right]\,.$$

Let us introduce the notation $s_{x} = \sin ( \pi x / N)$ and $c_{x} = \cos ( \pi x / N)$, which will simplify the next step. We can use the sine addition identity to write

$$ \sin\left( \frac{\pi (i+1) n}{N}\right)
= s_{in} c_n \pm c_{in} s_n\,.$$

Then, the RHS will equal

$$ 
(*) = 
Q_n(t) \left[ 
    s_{in} c_n + c_{in} s_n 
    +
    s_{in} c_n - c_{in} s_n 
    - 2 s_{in}
 \right] = Q_n(t) s_{in} (2c_n - 2)\,.$$

and through another trigonometric identity we can get to

$$ 2 c_n - 2 = 2 (\cos \left( \frac{\pi n}{N}\right) - 1) = - 4 \sin^2 \left( \frac{\pi n}{2N}\right)\,.$$

Then, if we denote $\omega_n = 2 \sin (\pi n / 2N)$, we get to the expression

$$ \sum_{n=1}^{N-1} \ddot{Q}_n (t) \sin\left( \frac{\pi i n}{N}\right) = -\sum_{n=1}^{N-1} \omega_n^2 Q_n(t) \sin\left( \frac{\pi i n}{N}\right)\,.$$
Alternatively, we can write this as

$$ \sum_{n=1}^{N-1} \left[ \ddot{Q}_n (t) + \omega_n^2 Q_n(t)  \right] \sin\left( \frac{\pi i n}{N}\right) = 0\,.$$

This should hold for all $i = 1\dots N-1$. We can interpret this as a matrix equation: $\sum_{n} U_{in}f_{n}=0$, where $U_{in} = \sin(\pi i n / N)$ while $f_{n} = \ddot{Q}_n (t) + \omega_n^2 Q_n(t)$. The matrix $U$ is invertible (check!), therefore the only solution to this system of equations is $f_{n}=0$ for all $n$.

## Hamiltonian and matrix approach

An alternative approach is the Hamiltonian one. We can write down the Hamiltonian of the problem in terms of the momenta $p_{i} = \dot{x}_{i}$ and coordinates $q_{i} = x_{i}$:
$$H = \frac{1}{2} \sum_{i=1}^{N-1} p_{i}^{2} + \frac{1}{2} \sum_{i=1}^{N} (x_{i} - x_{i-1})^{2}$$
We recover Hamilton’s equations as before:

$$\begin{align}
-\frac{\partial H}{\partial x_{i}} &= -( x_{i}- x_{i-1} ) - (x_{i}-x_{i+1}) = x_{i-1} +x_{i+1} - 2 x_{i}  \\
\frac{\partial H}{\partial \dot{x}_{i}} &= \dot{x}_{i}
\end{align}$$

In normal coordinates, the Hamiltonian reads:

$$H = \frac{1}{2} \sum_{k=1}^{N-1} \dot{Q}_{k}^{2} + \frac{1}{2} \sum_{k=1}^{N-1}  \omega _{k}^{2}Q_{k}^{2} = \sum_{k=1}^{N-1} E_{k}$$

Each mode is evolving independently!

In [2]:
import matplotlib.pyplot as plt
from project_scripts.simulations import non_linear_system
from pathlib import Path
from scipy import stats
import numpy as np

fname = Path('gaussian_y0.mp4')

if not fname.exists():
    y0 = np.concatenate((
        [0],
        stats.norm.pdf(np.linspace(-3, 2, num=2**5-1)),
        [0],
    ))

    non_linear_system(
        y0,
        periods=1.,
        video_path=Path('gaussian_y0.mp4'),
        label=', gaussian initial conditions',
        frames_per_period=400,
    )

    plt.close()

## Non-linear problem

Now, let us add some nonlinear terms to the Hamiltonian:

$$H = \frac{1}{2} \sum_{i=1}^{N-1} \dot{x}_{i}^{2} + \frac{1}{2} \sum_{i=1}^{N} \left[  (x_{i} - x_{i-1})^{2} + \frac{\alpha}{3}(x_{i}-x_{i-1})^{3} + \frac{\beta}{4} (x_{i}-x_{i-1})^{4}  \right]$$

The equations of motion now become
$$\ddot{x}_{i} = x_{i-1} +x_{i+1} - 2 x_{i} + \alpha \left[ (x_{i+1}-x_{i})^{2} - (x_{i}-x_{i-1})^{2} \right] + \beta \left[ (x_{i+1}-x_{i})^{3} - (x_{i}-x_{i-1})^{3} \right] $$

To add:

- normal mode description breakdown
- thermalization expectation
- approximate relation for the recurrence time

Useful resources:

- https://arxiv.org/pdf/nlin/0501053
- https://pubs.aip.org/aip/cha/article/15/1/015104/322124/The-Fermi-Pasta-Ulam-problem-Fifty-years-of

<!-- ```{python}
import numpy as np

N = 5

i = np.arange(N+1)
j = np.arange(N+1)

U = np.sqrt(2/N) * np.sin(i[:, np.newaxis] * j[np.newaxis, :] * np.pi / N)

print(np.round(U@U))
``` -->